In [ ]:
#| default_exp config

# config

> Pipeline configuration loaded from `manhualizer.yml` and/or environment variables.
>
> **Secrets** (API keys) are never stored in config files — they are read from environment variables only.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, Field
import yaml

## Output Config

In [ ]:
#| export
class OutputConfig(BaseModel):
    """Controls the format and dimensions of rendered panel images."""
    dir: str = "output"
    format: Literal["png", "jpg", "webp"] = "png"
    width: int = 1024
    height: int = 1536
    """Default 2:3 ratio — standard vertical manhua format."""
    aspect_ratio: str | None = None
    """Optional aspect ratio string (e.g. '9:16'). Overrides width/height when set."""

    def resolved_dimensions(self) -> tuple[int, int]:
        """Return (width, height) respecting aspect_ratio if set."""
        if not self.aspect_ratio:
            return self.width, self.height
        w_ratio, h_ratio = [int(x) for x in self.aspect_ratio.split(":")]
        return self.width, int(self.width * h_ratio / w_ratio)

## LLM Config

In [ ]:
#| export
class LLMConfig(BaseModel):
    """LLM settings. Model string uses litellm format: 'provider/model'.
    
    API keys are **not** stored here — they are read from environment variables
    (ANTHROPIC_API_KEY, OPENAI_API_KEY, GOOGLE_API_KEY, etc.).
    """
    model: str = "anthropic/claude-sonnet-4-6"
    temperature: float = 0.7
    max_tokens: int = 8192

## Renderer Configs (per model)

In [ ]:
#| export
class LoRAConfig(BaseModel):
    """A LoRA weight to apply during image generation (model-dependent)."""
    path: str
    """URL, local path, or model identifier — interpretation depends on the backend."""
    strength: float = 0.8
    trigger_word: str = ""

In [ ]:
#| export
class NanaBananaConfig(BaseModel):
    """Config for the NanaBanana (Google Imagen) backend."""
    model: str = "imagen-3.0-generate-002"
    location: str = "us-central1"
    """API key read from GOOGLE_API_KEY env var."""

In [ ]:
#| export
class ChatGPTImageConfig(BaseModel):
    """Config for the OpenAI image generation backend."""
    model: str = "dall-e-3"
    quality: Literal["standard", "hd"] = "standard"
    """API key read from OPENAI_API_KEY env var."""

In [ ]:
#| export
class SeedreamConfig(BaseModel):
    """Config for the Seedream model (generic Replicate deployment)."""
    model_ref: str = ""
    """Model reference (e.g. Replicate model string). Set to your deployment."""

In [ ]:
#| export
class Seedream45Config(BaseModel):
    """Config for Seedream 4.5 (bytedance/seedream-4.5 on Replicate).
    
    The model endpoint is fixed — no model_ref needed.
    All generation parameters are derived from OutputConfig and the panel prompt.
    """
    pass

In [ ]:
#| export
class FluxKleinConfig(BaseModel):
    """Config for the Flux Klein 9B model."""
    model_ref: str = ""
    """Model reference (e.g. Replicate model string). Set to your deployment."""

In [ ]:
#| export
class ComfyUIConfig(BaseModel):
    """Config for a local ComfyUI server."""
    base_url: str = "http://127.0.0.1:8188"
    workflow_template_path: str = ""
    """Path to a ComfyUI workflow JSON exported in API format."""
    poll_interval: float = 2.0
    timeout: float = 120.0

In [ ]:
#| export
class RendererConfig(BaseModel):
    """Image generation settings: which model to use and its parameters."""
    model: str = "flux-klein"
    """Key into the model registry. One of: nanobanana, chatgpt-image, seedream, seedream-4.5, flux-klein, comfyui."""
    loras: list[LoRAConfig] = Field(default_factory=list)
    """LoRA weights to apply. Silently ignored for models that don't support LoRA."""
    nanobanana: NanaBananaConfig = Field(default_factory=NanaBananaConfig)
    chatgpt_image: ChatGPTImageConfig = Field(default_factory=ChatGPTImageConfig)
    seedream: SeedreamConfig = Field(default_factory=SeedreamConfig)
    seedream_45: Seedream45Config = Field(default_factory=Seedream45Config)
    flux_klein: FluxKleinConfig = Field(default_factory=FluxKleinConfig)
    comfyui: ComfyUIConfig = Field(default_factory=ComfyUIConfig)

## Pipeline Config

In [ ]:
#| export
class PipelineConfig(BaseModel):
    """Top-level configuration for a manhualizer run.
    
    Can be loaded from a `manhualizer.yml` file and/or overridden via CLI flags.
    """
    llm: LLMConfig = Field(default_factory=LLMConfig)
    renderer: RendererConfig = Field(default_factory=RendererConfig)
    output: OutputConfig = Field(default_factory=OutputConfig)
    template: str = "default"
    """Name of the prompt template set to use (built-in or from custom_templates_dir)."""
    custom_templates_dir: str | None = None
    """Path to a directory of custom template YAML files."""
    panels_per_scene: int = 4
    max_chunk_tokens: int = 3000
    run_validation: bool = False
    """Run the optional LLM validation step (adds latency and cost)."""
    resume: bool = True
    """Skip steps whose output files already exist."""

## Config Loader

In [ ]:
#| export
def load_config(path: Path | str | None = None, **overrides) -> PipelineConfig:
    """Load PipelineConfig from a YAML file, then apply any keyword overrides.
    
    Lookup order:
    1. Defaults (PipelineConfig field defaults)
    2. YAML file at `path` (or `manhualizer.yml` in CWD if `path` is None and the file exists)
    3. `**overrides` keyword arguments (from CLI flags)
    
    API keys are **never** stored in YAML — set them as environment variables.
    """
    data: dict = {}

    if path is None:
        default_path = Path("manhualizer.yml")
        if default_path.exists():
            path = default_path

    if path is not None:
        with open(path) as f:
            data = yaml.safe_load(f) or {}

    # Apply CLI overrides as flat keys → nested structure
    _apply_overrides(data, overrides)

    return PipelineConfig.model_validate(data)


def _apply_overrides(data: dict, overrides: dict) -> None:
    """Merge flat CLI overrides into nested config dict. None values are ignored."""
    mapping = {
        "output_dir":      ("output", "dir"),
        "format":          ("output", "format"),
        "width":           ("output", "width"),
        "height":          ("output", "height"),
        "aspect_ratio":    ("output", "aspect_ratio"),
        "llm_model":       ("llm", "model"),
        "model":           ("renderer", "model"),
        "template":        ("template",),
        "resume":          ("resume",),
        "run_validation":  ("run_validation",),
        "style_prefix":    None,  # handled via template system, not config
    }
    for key, value in overrides.items():
        if value is None:
            continue
        path = mapping.get(key)
        if path is None:
            continue
        node = data
        for part in path[:-1]:
            node = node.setdefault(part, {})
        node[path[-1]] = value

## Basic Tests

In [ ]:
# Default config
cfg = PipelineConfig()
assert cfg.renderer.model == "flux-klein"
assert cfg.output.format == "png"
assert cfg.output.resolved_dimensions() == (1024, 1536)

# Aspect ratio override
out = OutputConfig(width=1024, aspect_ratio="16:9")
w, h = out.resolved_dimensions()
assert w == 1024 and h == 576

# load_config with overrides
cfg2 = load_config(output_dir="/tmp/out", model="nanobanana")
assert cfg2.output.dir == "/tmp/out"
assert cfg2.renderer.model == "nanobanana"
print("Config OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()